# Long Document

In [1]:
long_document = """
University Academic Handbook

Attendance Policy:
Students must maintain at least 75% attendance in every subject to be eligible for semester examinations.
Students with attendance below 75% may not be allowed to write final exams unless special approval is granted.

Hostel Rules:
Hostel gates close at 9 PM on weekdays and 10 PM on weekends.
Students must carry their ID cards while entering the hostel.

Fee Refund Policy:
The refund deadline for semester fees is 15 days after the start of classes.
No refund will be issued after this deadline except under extraordinary circumstances.

Library Policy:
Students can borrow up to 4 books at a time.
Books must be returned within 14 days to avoid a fine.

Examination Rules:
Students must carry their hall ticket to the exam hall.
Electronic devices are not allowed during exams.
"""

# Paragraph-based chunking

In [2]:
chunks = [chunk.strip() for chunk in long_document.split("\n\n") if chunk.strip()]

In [3]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}:")
    print(chunk)
    print("-" * 50)

Chunk 0:
University Academic Handbook
--------------------------------------------------
Chunk 1:
Attendance Policy:
Students must maintain at least 75% attendance in every subject to be eligible for semester examinations.
Students with attendance below 75% may not be allowed to write final exams unless special approval is granted.
--------------------------------------------------
Chunk 2:
Hostel Rules:
Hostel gates close at 9 PM on weekdays and 10 PM on weekends.
Students must carry their ID cards while entering the hostel.
--------------------------------------------------
Chunk 3:
Fee Refund Policy:
The refund deadline for semester fees is 15 days after the start of classes.
No refund will be issued after this deadline except under extraordinary circumstances.
--------------------------------------------------
Chunk 4:
Library Policy:
Students can borrow up to 4 books at a time.
Books must be returned within 14 days to avoid a fine.
-------------------------------------------------

# Embeddings

In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = model.encode(chunks)

c:\Tanisha\College\RAG-from-scratch\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4770.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# FAISS

In [5]:
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings, dtype="float32"))

# Query

In [6]:
query = "What is the refund deadline for semester fees?"
query_embedding = model.encode([query])

k = 2
distances, indices = index.search(np.array(query_embedding, dtype="float32"), k)

In [7]:
print("Query:", query)
print("\nTop retrieved chunks:\n")

for rank, i in enumerate(indices[0], start=1):
    print(f"Rank {rank}:")
    print(chunks[i])
    print(f"Distance: {distances[0][rank-1]:.4f}")
    print()

Query: What is the refund deadline for semester fees?

Top retrieved chunks:

Rank 1:
Fee Refund Policy:
The refund deadline for semester fees is 15 days after the start of classes.
No refund will be issued after this deadline except under extraordinary circumstances.
Distance: 0.2642

Rank 2:
Library Policy:
Students can borrow up to 4 books at a time.
Books must be returned within 14 days to avoid a fine.
Distance: 1.1862



In [8]:
test_queries = [
    "What attendance is required for semester exams?",
    "When do hostel gates close?",
    "How many books can students borrow?",
    "Are electronic devices allowed in exams?"
]

for query in test_queries:
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding, dtype="float32"), 1)
    
    print(f"\nQuery: {query}")
    print("Top retrieved chunk:")
    print(chunks[indices[0][0]])
    print(f"Distance: {distances[0][0]:.4f}")
    print("-" * 60)


Query: What attendance is required for semester exams?
Top retrieved chunk:
Attendance Policy:
Students must maintain at least 75% attendance in every subject to be eligible for semester examinations.
Students with attendance below 75% may not be allowed to write final exams unless special approval is granted.
Distance: 0.5294
------------------------------------------------------------

Query: When do hostel gates close?
Top retrieved chunk:
Hostel Rules:
Hostel gates close at 9 PM on weekdays and 10 PM on weekends.
Students must carry their ID cards while entering the hostel.
Distance: 0.3715
------------------------------------------------------------

Query: How many books can students borrow?
Top retrieved chunk:
Library Policy:
Students can borrow up to 4 books at a time.
Books must be returned within 14 days to avoid a fine.
Distance: 0.5624
------------------------------------------------------------

Query: Are electronic devices allowed in exams?
Top retrieved chunk:
Examina